# 03 — Bad Channel Detection

Detects bad channels automatically (LOF algorithm) and opens an interactive plot
for manual confirmation. Bad channels are interpolated and the data is re-referenced
to the average.

**Requires:** Qt5 backend for interactive plots (`matplotlib.use('Qt5Agg')`)

**Input:** `<subject>_preprocessed_raw.fif`  
**Output:** `<subject>_preprocessed_clean_raw.fif`, `<subject>_bad_channels.json`

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Qt5Agg')

from eeg_toolkit import load_config, find_subjects, inspect_subject_bads, inspect_all_subjects

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects to inspect: {len(subjects)}")
print(subjects)

In [ ]:
# ── Inspect one subject (test run) ──
# threshold: LOF sensitivity (lower = more aggressive, typical: 1.5–2.0)
# Interactive plot:
#   - Click channel names to toggle bad status
#   - Inspect the PSD for spectral outliers
#   - Close BOTH windows when done to save
#   - The temporal one is the one that ends up selecting the channels to interpolate
inspect_subject_bads(cfg, subjects[0], threshold=1.5, overwrite=False, verbose=True)

In [ ]:
# ── Inspect all remaining subjects ──
summary = inspect_all_subjects(cfg, threshold=1.5, overwrite=False, verbose=True)

In [ ]:
# ── Summary: bad channels per subject ──
from eeg_toolkit import load_status
import pandas as pd

df_status = load_status(cfg)
summary = df_status[['subject', 'n_bad_channels', 'bad_channel_names']].copy()
summary['n_bad_channels'] = pd.to_numeric(summary['n_bad_channels'], errors='coerce')

print("=== Bad channels per subject ===")
print(summary.to_string(index=False))
print(f"\nMean:   {summary['n_bad_channels'].mean():.2f}")
print(f"Median: {summary['n_bad_channels'].median():.0f}")
print(f"Max:    {summary['n_bad_channels'].max():.0f}")